In [1]:

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
import zipfile
import yaml

# Google Drive paths
DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"
DRIVE_ZIP = os.path.join(
    DRIVE_ROOT, "hard_mined_soccernet_ball_yolo.zip"
)
CHAMPION = os.path.join(
    DRIVE_ROOT, "ball_detector_yolo11n_5ep", "weights", "best.pt"
)

# Colab local paths
LOCAL_ZIP = "/content/hard_mined_soccernet_ball_yolo.zip"
DATASET_ROOT = "/content/hard_mined_soccernet_ball_yolo"

assert os.path.isfile(DRIVE_ZIP), f"Hard-mined ZIP missing: {DRIVE_ZIP}"
assert os.path.isfile(CHAMPION), f"Champion checkpoint missing: {CHAMPION}"

# Copy ZIP from Drive to local Colab storage
if (
    not os.path.isfile(LOCAL_ZIP)
    or os.path.getsize(LOCAL_ZIP) != os.path.getsize(DRIVE_ZIP)
):
    print("Copying hard-mined dataset from Drive...")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
else:
    print("Local ZIP already exists with matching size.")

print(f"ZIP size: {os.path.getsize(LOCAL_ZIP) / (1024**3):.2f} GB")

# Extract archive
EXPECTED_YAML = os.path.join(DATASET_ROOT, "data.yaml")

if not os.path.isfile(EXPECTED_YAML):
    os.makedirs(DATASET_ROOT, exist_ok=True)

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        bad_file = zf.testzip()
        assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

        print(f"Extracting {len(zf.infolist())} files...")
        zf.extractall(DATASET_ROOT)
else:
    print("Dataset already extracted.")

# Verify the archive's expected structure
for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_ROOT, split, "images")
    label_dir = os.path.join(DATASET_ROOT, split, "labels")

    assert os.path.isdir(image_dir), f"Missing {image_dir}"
    assert os.path.isdir(label_dir), f"Missing {label_dir}"

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]
    labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]

    print(f"{split}: {len(images)} images, {len(labels)} label files")
    assert len(images) == 1500, f"Unexpected {split} image count"

# Create a YOLO dataset YAML for this archive's folder structure
DATA_YAML = os.path.join(DATASET_ROOT, "data.yaml")
config = {
    "path": DATASET_ROOT,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "ball"}
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\nDataset prepared:", DATA_YAML)
print("Champion checkpoint:", CHAMPION)
print(f"Champion size: {os.path.getsize(CHAMPION)/(1024**2):.2f} MB")


Mounted at /content/drive
Copying hard-mined dataset from Drive...
ZIP size: 0.96 GB
Extracting 9004 files...
train: 1500 images, 1500 label files
val: 1500 images, 1500 label files
test: 1500 images, 1500 label files

Dataset prepared: /content/hard_mined_soccernet_ball_yolo/data.yaml
Champion checkpoint: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt
Champion size: 5.27 MB


In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.4 MB/s eta 0:00:00


In [4]:

import torch
from ultralytics import YOLO

DATA_YAML = "/content/hard_mined_soccernet_ball_yolo/data.yaml"
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

assert torch.cuda.is_available(), "Enable the Colab GPU first."

model = YOLO(CHAMPION)

print("GPU:", torch.cuda.get_device_name(0))
print("Resolution: 1280")
print("Benchmarking existing champion on hard validation set...")

metrics = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=1280,
    batch=16,
    device=0,
    workers=2,
    project="/content/athletiq_runs",
    name="baseline_hard_val_1280",
    plots=True,
    verbose=True
)

baseline = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map)
}

print("\nBASELINE — 1280 ONLY")
for name, value in baseline.items():
    print(f"{name}: {value:.4f} ({value*100:.2f}%)")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU: Tesla T4
Resolution: 1280
Benchmarking existing champion on hard validation set...
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2971.0±412.3 MB/s, size: 284.2 KB)
val: Scanning /content/hard_mined_soccernet_ball_yolo/val/labels... 1500 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.1Kit/s 0.7s
val: New cache created: /content/hard_mined_soccernet_ball_yolo/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 2.4it/s 39.7s
         

In [6]:

import os
import torch
from ultralytics import YOLO

DATA_YAML = "/content/hard_mined_soccernet_ball_yolo/data.yaml"
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

RUN_PROJECT = "/content/athletiq_runs"
RUN_NAME = "hard_finetune_yolo11n_1280"

assert torch.cuda.is_available()
assert os.path.isfile(DATA_YAML)
assert os.path.isfile(CHAMPION)

# Always fine-tune from the original champion.
model = YOLO(CHAMPION)

results = model.train(
    data=DATA_YAML,
    imgsz=1280,
    epochs=20,
    patience=5,
    batch=20,
    device=0,
    workers=2,

    # Controlled fine-tuning learning rate
    lr0=0.0005,
    lrf=0.01,
    warmup_epochs=2,
    cos_lr=True,

    # Regularization
    weight_decay=0.0005,

    # Realistic augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=3.0,
    translate=0.05,
    scale=0.25,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    close_mosaic=5,
    mixup=0.0,

    # Reproducibility and checkpoint management
    seed=42,
    deterministic=True,
    amp=True,
    pretrained=True,
    save=True,
    save_period=5,
    plots=True,
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    verbose=True
)

print("\nTraining finished.")
print("Run directory:", os.path.join(RUN_PROJECT, RUN_NAME))
print("Best checkpoint:", os.path.join(
    RUN_PROJECT, RUN_NAME, "weights", "best.pt"
))


Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/hard_mined_soccernet_ball_yolo/data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt, momentum=0.937, mosaic=0.5

In [7]:

import os
import yaml

original_root = "/content/soccernet_ball_yolo/soccernet_ball_yolo"
hard_root = "/content/hard_mined_soccernet_ball_yolo"
run_dir = "/content/athletiq_runs/hard_finetune_yolo11n_1280"

print("ORIGINAL DATASET")
for split in ["train", "val", "test"]:
    path = os.path.join(original_root, "images", split)
    print(f"{split}: {os.path.isdir(path)}", path)

print("\nHARD-MINED DATASET")
for split in ["train", "val", "test"]:
    path = os.path.join(hard_root, split, "images")
    count = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f"{split}: {count} images")

print("\nTRAINING RUN")
args_path = os.path.join(run_dir, "args.yaml")
if os.path.isfile(args_path):
    with open(args_path, "r") as f:
        args = yaml.safe_load(f)

    for key in [
        "imgsz", "batch", "optimizer", "lr0", "lrf",
        "warmup_epochs", "cos_lr", "weight_decay",
        "mosaic", "scale", "dropout"
    ]:
        print(f"{key}: {args.get(key)}")
else:
    print("Training args.yaml not found.")


ORIGINAL DATASET
train: False /content/soccernet_ball_yolo/soccernet_ball_yolo/images/train
val: False /content/soccernet_ball_yolo/soccernet_ball_yolo/images/val
test: False /content/soccernet_ball_yolo/soccernet_ball_yolo/images/test

HARD-MINED DATASET
train: 1500 images
val: 1500 images
test: 1500 images

TRAINING RUN
imgsz: 1280
batch: 20
optimizer: auto
lr0: 0.0005
lrf: 0.01
warmup_epochs: 2
cos_lr: True
weight_decay: 0.0005
mosaic: 0.5
scale: 0.25
dropout: 0.0


In [8]:

import os
from ultralytics import YOLO

CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)
FAILED_BEST = (
    "/content/athletiq_runs/hard_finetune_yolo11n_1280/"
    "weights/best.pt"
)
DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"

for name, path in [
    ("Original champion", CHAMPION),
    ("Previous fine-tuned best", FAILED_BEST),
]:
    print(f"\n{name}:")
    print("Exists:", os.path.isfile(path))
    if os.path.isfile(path):
        print("Size MB:", round(os.path.getsize(path) / 1024**2, 2))

print("\nPossible backup ZIPs in Drive:")
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in {
        ".git", "node_modules", ".venv"
    }]
    for filename in files:
        if filename.lower().endswith(".zip") and (
            "soccernet" in filename.lower()
            or "football" in filename.lower()
        ):
            print(os.path.join(root, filename))



Original champion:
Exists: True
Size MB: 5.27

Previous fine-tuned best:
Exists: True
Size MB: 5.27

Possible backup ZIPs in Drive:
/content/drive/MyDrive/AthletiQ/hard_mined_soccernet_ball_yolo.zip


In [9]:

import os

candidates = [
    "/content/soccernet_ball_yolo.zip",
    "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml",
    "/content/soccernet_ball_yolo/soccernet_ball_yolo/images/train",
    "/content/hard_mined_soccernet_ball_yolo.zip",
]

for path in candidates:
    if os.path.exists(path):
        if os.path.isfile(path):
            size_gb = os.path.getsize(path) / (1024**3)
            print(f"[FOUND FILE] {path} ({size_gb:.2f} GB)")
        else:
            print(f"[FOUND DIR]  {path}")
    else:
        print(f"[MISSING]    {path}")


[MISSING]    /content/soccernet_ball_yolo.zip
[MISSING]    /content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml
[MISSING]    /content/soccernet_ball_yolo/soccernet_ball_yolo/images/train
[FOUND FILE] /content/hard_mined_soccernet_ball_yolo.zip (0.96 GB)


In [10]:

import os
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"
FAILED_BEST = (
    "/content/athletiq_runs/hard_finetune_yolo11n_1280/"
    "weights/best.pt"
)
BACKUP = os.path.join(
    DRIVE_ROOT, "models", "hard_finetune_candidate_epoch1.pt"
)

assert os.path.isfile(FAILED_BEST), "Previous candidate checkpoint missing."

os.makedirs(os.path.dirname(BACKUP), exist_ok=True)
shutil.copy2(FAILED_BEST, BACKUP)

assert os.path.getsize(BACKUP) == os.path.getsize(FAILED_BEST)

print("Previous candidate backed up:", BACKUP)
print("Production champion remains untouched.")


Previous candidate backed up: /content/drive/MyDrive/AthletiQ/models/hard_finetune_candidate_epoch1.pt
Production champion remains untouched.


In [11]:

import os
import shutil
import yaml
import torch
from ultralytics import YOLO

# Dataset and original champion
DATA_YAML = "/content/hard_mined_soccernet_ball_yolo/data.yaml"
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

# New run and separate Drive checkpoint
PROJECT = "/content/athletiq_runs"
RUN_NAME = "hard_finetune_yolo11n_1280_controlled"
RUN_DIR = os.path.join(PROJECT, RUN_NAME)
BEST = os.path.join(RUN_DIR, "weights", "best.pt")

DRIVE_SAVE = (
    "/content/drive/MyDrive/AthletiQ/models/"
    "athletiq_ball_detector_hard_finetuned_candidate.pt"
)

assert torch.cuda.is_available(), "Enable the Tesla T4 GPU."
assert os.path.isfile(DATA_YAML), "Hard-mined dataset YAML missing."
assert os.path.isfile(CHAMPION), "Original champion missing."

# Start from the known best production model.
model = YOLO(CHAMPION)

print("GPU:", torch.cuda.get_device_name(0))
print("Starting checkpoint:", CHAMPION)
print("Resolution: 1280")
print("Batch size: 4")
print("Optimizer: explicitly set to AdamW")
print("Learning rate: 0.0005 -> 0.000005")

# Controlled fine-tuning
model.train(
    data=DATA_YAML,
    imgsz=1280,
    epochs=20,
    patience=8,
    batch=20,
    device=0,
    workers=2,

    # Force the intended optimizer and learning-rate schedule
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    warmup_epochs=2,
    cos_lr=True,
    momentum=0.9,
    weight_decay=0.0005,

    # Moderate, realistic augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=2.0,
    translate=0.04,
    scale=0.20,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.25,
    close_mosaic=5,
    mixup=0.0,
    copy_paste=0.0,
    dropout=0.0,

    # Stability and reproducibility
    pretrained=True,
    amp=True,
    seed=42,
    deterministic=True,

    save=True,
    save_period=5,
    plots=True,
    val=True,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    verbose=True
)

# Verify that the intended settings were recorded
ARGS_PATH = os.path.join(RUN_DIR, "args.yaml")
if os.path.isfile(ARGS_PATH):
    with open(ARGS_PATH, "r") as f:
        args = yaml.safe_load(f)

    print("\nRECORDED TRAINING SETTINGS")
    for key in [
        "optimizer", "lr0", "lrf", "warmup_epochs",
        "cos_lr", "weight_decay", "imgsz", "batch"
    ]:
        print(f"{key}: {args.get(key)}")

    assert str(args.get("optimizer")).lower() == "adamw"
    assert abs(float(args["lr0"]) - 0.0005) < 1e-9

# Save the best checkpoint to Drive without touching Stage 1.
assert os.path.isfile(BEST), f"Best checkpoint not found: {BEST}"

os.makedirs(os.path.dirname(DRIVE_SAVE), exist_ok=True)
shutil.copy2(BEST, DRIVE_SAVE)

assert os.path.getsize(DRIVE_SAVE) == os.path.getsize(BEST)

print("\nCandidate saved successfully:")
print(DRIVE_SAVE)
print(f"Size: {os.path.getsize(DRIVE_SAVE)/(1024**2):.2f} MB")
print("\nStage 1 production checkpoint remains unchanged.")


GPU: Tesla T4
Starting checkpoint: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt
Resolution: 1280
Batch size: 4
Optimizer: explicitly set to AdamW
Learning rate: 0.0005 -> 0.000005
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/hard_mined_soccernet_ball_yolo/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.35, hsv_v=0.25, imgsz=1280, iou=0

In [12]:
import os
import json
import csv
import shutil
from datetime import datetime
from pathlib import Path

import torch
from ultralytics import YOLO

# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_YAML = (
    "/content/hard_mined_soccernet_ball_yolo/data.yaml"
)

ORIGINAL_CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

FINETUNED_CANDIDATE = (
    "/content/drive/MyDrive/AthletiQ/models/"
    "athletiq_ball_detector_hard_finetuned_candidate.pt"
)

DRIVE_ROOT = Path("/content/drive/MyDrive/AthletiQ")

# All experiments for this finalization go in this folder.
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / "ball_detector_finalization"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

# Fixed resolution for evaluation and deployment.
IMAGE_SIZE = 1280
BATCH_SIZE = 4
DEVICE = 0

# ============================================================
# 2. CHECK REQUIREMENTS
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU is not available."
assert os.path.isfile(DATA_YAML), f"Dataset YAML missing: {DATA_YAML}"
assert os.path.isfile(ORIGINAL_CHAMPION), "Original champion checkpoint missing."
assert os.path.isfile(FINETUNED_CANDIDATE), "Fine-tuned candidate checkpoint missing."

print("GPU:", torch.cuda.get_device_name(0))
print("Resolution:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Validation dataset:", DATA_YAML)
print("Experiment folder:", EXPERIMENT_ROOT)

# ============================================================
# 3. CREATE A UNIQUE EXPERIMENT FOLDER
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = EXPERIMENT_ROOT / f"final_comparison_{timestamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("\nExperiment ID:", RUN_DIR.name)

# ============================================================
# 4. EVALUATE BOTH MODELS ON THE SAME VALIDATION SET
# ============================================================

models_to_test = {
    "original_champion": ORIGINAL_CHAMPION,
    "finetuned_candidate": FINETUNED_CANDIDATE,
}

results_summary = {}

for model_name, checkpoint in models_to_test.items():
    print("\n" + "=" * 65)
    print("Evaluating:", model_name)
    print("Checkpoint:", checkpoint)
    print("=" * 65)

    model = YOLO(checkpoint)

    metrics = model.val(
        data=DATA_YAML,
        split="val",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=2,
        plots=True,
        save_json=False,
        project=str(RUN_DIR),
        name=model_name,
        exist_ok=False,
        verbose=True,
    )

    box = metrics.box

    results_summary[model_name] = {
        "checkpoint": checkpoint,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "precision": float(box.mp),
        "recall": float(box.mr),
        "map50": float(box.map50),
        "map50_95": float(box.map),
        "preprocess_ms_per_image": float(
            metrics.speed.get("preprocess", 0)
        ),
        "inference_ms_per_image": float(
            metrics.speed.get("inference", 0)
        ),
        "postprocess_ms_per_image": float(
            metrics.speed.get("postprocess", 0)
        ),
    }

    print(f"\n{model_name} results")
    print(f"Precision:    {box.mp:.4f}")
    print(f"Recall:       {box.mr:.4f}")
    print(f"mAP@50:       {box.map50:.4f}")
    print(f"mAP@50-95:    {box.map:.4f}")
    print(f"Inference:    {metrics.speed.get('inference', 0):.2f} ms/image")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 5. SAVE COMPARISON REPORTS
# ============================================================

with open(RUN_DIR / "metrics.json", "w") as f:
    json.dump(results_summary, f, indent=4)

csv_path = RUN_DIR / "comparison.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "model",
            "precision",
            "recall",
            "map50",
            "map50_95",
            "preprocess_ms_per_image",
            "inference_ms_per_image",
            "postprocess_ms_per_image",
            "image_size",
            "batch_size",
            "checkpoint",
        ],
    )
    writer.writeheader()

    for name, values in results_summary.items():
        writer.writerow({
            "model": name,
            **values,
        })

# ============================================================
# 6. SELECT THE CHAMPION
#    Primary metric: mAP@50-95.
#    Require mAP@50 not to regress.
# ============================================================

original = results_summary["original_champion"]
candidate = results_summary["finetuned_candidate"]

candidate_wins = (
    candidate["map50_95"] > original["map50_95"]
    and candidate["map50"] >= original["map50"]
)

if candidate_wins:
    winner_name = "finetuned_candidate"
    winner_checkpoint = FINETUNED_CANDIDATE
else:
    winner_name = "original_champion"
    winner_checkpoint = ORIGINAL_CHAMPION

# Keep the original and candidate files untouched.
CHAMPION_DIR = DRIVE_ROOT / "models"
CHAMPION_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CHAMPION = (
    CHAMPION_DIR / "athletiq_ball_detector_champion_1280.pt"
)

shutil.copy2(winner_checkpoint, FINAL_CHAMPION)

decision = {
    "selected_model": winner_name,
    "selected_checkpoint": str(FINAL_CHAMPION),
    "selection_rule": (
        "Higher hard-validation mAP@50-95, with mAP@50 "
        "not lower than the original champion."
    ),
    "image_size": IMAGE_SIZE,
    "validation_split": "hard-mined validation",
    "warning": (
        "Selection is based on the 1500-image hard validation subset. "
        "Evaluate the untouched hard test split once for final reporting."
    ),
    "metrics": results_summary,
}

with open(RUN_DIR / "final_decision.json", "w") as f:
    json.dump(decision, f, indent=4)

# Copy the chosen checkpoint into the experiment folder too.
shutil.copy2(
    winner_checkpoint,
    RUN_DIR / "selected_champion.pt"
)

# Save a human-readable report.
with open(RUN_DIR / "README.txt", "w") as f:
    f.write("AthletiQ Ball Detector - Final Comparison\n")
    f.write("=" * 45 + "\n")
    f.write(f"Experiment: {RUN_DIR.name}\n")
    f.write(f"Image size: {IMAGE_SIZE}\n")
    f.write(f"Validation split: hard-mined validation\n\n")

    for name, values in results_summary.items():
        f.write(f"{name}\n")
        f.write(f"  Precision: {values['precision']:.4f}\n")
        f.write(f"  Recall: {values['recall']:.4f}\n")
        f.write(f"  mAP@50: {values['map50']:.4f}\n")
        f.write(f"  mAP@50-95: {values['map50_95']:.4f}\n")
        f.write(
            f"  Inference ms/image: "
            f"{values['inference_ms_per_image']:.2f}\n\n"
        )

    f.write(f"Selected model: {winner_name}\n")
    f.write(f"Final checkpoint: {FINAL_CHAMPION}\n")

# ============================================================
# 7. DISPLAY FINAL DECISION
# ============================================================

print("\n" + "=" * 65)
print("FINAL MODEL SELECTION")
print("=" * 65)

for name, values in results_summary.items():
    print(
        f"{name:22s} | "
        f"P={values['precision']:.4f} | "
        f"R={values['recall']:.4f} | "
        f"mAP50={values['map50']:.4f} | "
        f"mAP50-95={values['map50_95']:.4f}"
    )

print("\nSelected model:", winner_name)
print("Final checkpoint:", FINAL_CHAMPION)
print("Experiment artifacts:", RUN_DIR)
print("Comparison CSV:", csv_path)
print("Metrics JSON:", RUN_DIR / "metrics.json")
print("Decision report:", RUN_DIR / "final_decision.json")
print("\nOriginal champion checkpoint remains unchanged.")

GPU: Tesla T4
Resolution: 1280
Batch size: 4
Validation dataset: /content/hard_mined_soccernet_ball_yolo/data.yaml
Experiment folder: /content/drive/MyDrive/AthletiQ/experiments/ball_detector_finalization

Experiment ID: final_comparison_20261018_163815

Evaluating: original_champion
Checkpoint: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2396.0±628.9 MB/s, size: 247.2 KB)
val: Scanning /content/hard_mined_soccernet_ball_yolo/val/labels.cache... 1500 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 262.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 375/375 10.0it/s 37.4s
                   all       1500       1868      0.746      0.699      0.671      0.268


In [13]:
import os
import json
import csv
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

# Final selected champion
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/models/"
    "athletiq_ball_detector_champion_1280.pt"
)

DATA_YAML = "/content/hard_mined_soccernet_ball_yolo/data.yaml"

EXPERIMENT_ROOT = Path(
    "/content/drive/MyDrive/AthletiQ/experiments/"
    "ball_detector_finalization"
)

assert os.path.isfile(CHAMPION), f"Missing champion: {CHAMPION}"
assert os.path.isfile(DATA_YAML), f"Missing dataset YAML: {DATA_YAML}"

# Keep final test results separate from validation experiments.
TEST_DIR = EXPERIMENT_ROOT / (
    "final_test_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
TEST_DIR.mkdir(parents=True, exist_ok=False)

print("Evaluating final champion on TEST split")
print("Checkpoint:", CHAMPION)
print("Image size: 1280")
print("Results folder:", TEST_DIR)

model = YOLO(CHAMPION)

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=1280,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    project=str(TEST_DIR),
    name="champion_test_metrics",
    exist_ok=False,
    verbose=True,
)

box = metrics.box

report = {
    "model": "AthletiQ Ball Detector Champion",
    "checkpoint": CHAMPION,
    "split": "test",
    "image_size": 1280,
    "precision": float(box.mp),
    "recall": float(box.mr),
    "map50": float(box.map50),
    "map50_95": float(box.map),
    "preprocess_ms_per_image": float(
        metrics.speed.get("preprocess", 0)
    ),
    "inference_ms_per_image": float(
        metrics.speed.get("inference", 0)
    ),
    "postprocess_ms_per_image": float(
        metrics.speed.get("postprocess", 0)
    ),
}

with open(TEST_DIR / "test_metrics.json", "w") as f:
    json.dump(report, f, indent=4)

with open(TEST_DIR / "test_metrics.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(report.keys()))
    writer.writeheader()
    writer.writerow(report)

print("\n========== FINAL TEST RESULTS ==========")
print(f"Precision:       {box.mp:.4f} ({box.mp * 100:.2f}%)")
print(f"Recall:          {box.mr:.4f} ({box.mr * 100:.2f}%)")
print(f"mAP@50:          {box.map50:.4f} ({box.map50 * 100:.2f}%)")
print(f"mAP@50-95:       {box.map:.4f} ({box.map * 100:.2f}%)")
print(f"Inference:       {metrics.speed.get('inference', 0):.2f} ms/image")
print("Saved metrics:", TEST_DIR / "test_metrics.json")
print("Saved CSV:", TEST_DIR / "test_metrics.csv")

Evaluating final champion on TEST split
Checkpoint: /content/drive/MyDrive/AthletiQ/models/athletiq_ball_detector_champion_1280.pt
Image size: 1280
Results folder: /content/drive/MyDrive/AthletiQ/experiments/ball_detector_finalization/final_test_20261018_164028
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 63.6±24.3 MB/s, size: 221.3 KB)
val: Scanning /content/hard_mined_soccernet_ball_yolo/test/labels... 1500 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 327.0it/s 4.6s
val: New cache created: /content/hard_mined_soccernet_ball_yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 375/375 10.9it/s 34.6s
                   all       1500       1812       0.59      0.367      0.365      0.123
Speed: 3.2ms preprocess, 10.8ms infe

In [14]:
import os
import json
from pathlib import Path
from ultralytics import YOLO

ORIGINAL_CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

DATA_YAML = "/content/hard_mined_soccernet_ball_yolo/data.yaml"

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/AthletiQ/experiments/"
    "ball_detector_finalization/original_champion_test"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert os.path.isfile(ORIGINAL_CHAMPION)
assert os.path.isfile(DATA_YAML)

model = YOLO(ORIGINAL_CHAMPION)

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=1280,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    project=str(OUTPUT_DIR),
    name="original_champion_test_metrics",
    exist_ok=True,
    verbose=True,
)

report = {
    "model": "original_champion",
    "image_size": 1280,
    "split": "test",
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "inference_ms_per_image": float(
        metrics.speed.get("inference", 0)
    ),
}

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(report, f, indent=4)

print("\n===== ORIGINAL CHAMPION: TEST RESULTS =====")
print(f"Precision:    {report['precision'] * 100:.2f}%")
print(f"Recall:       {report['recall'] * 100:.2f}%")
print(f"mAP@50:       {report['map50'] * 100:.2f}%")
print(f"mAP@50-95:    {report['map50_95'] * 100:.2f}%")
print(f"Inference:    {report['inference_ms_per_image']:.2f} ms/image")
print("Saved:", OUTPUT_DIR / "metrics.json")

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1580.9±369.1 MB/s, size: 212.2 KB)
val: Scanning /content/hard_mined_soccernet_ball_yolo/test/labels.cache... 1500 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 349.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 375/375 10.5it/s 35.6s
                   all       1500       1812      0.551      0.459      0.422      0.141
Speed: 3.6ms preprocess, 10.8ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/drive/.shortcut-targets-by-id/1TZjfXqSAdZ4iG5owY9uEYWULp7VyS0ih/AthletiQ/experiments/ball_detector_finalization/original_champion_test/original_champion_test_metrics

===== ORIGINAL CHAMPION: TEST RESULTS =====
Precision:    55.13%
Recall:       45.92%
mAP@50: 

In [15]:
import json
import shutil
from pathlib import Path
from datetime import datetime

DRIVE = Path("/content/drive/MyDrive/AthletiQ")

ORIGINAL = (
    DRIVE / "ball_detector_yolo11n_5ep/weights/best.pt"
)
CANDIDATE = (
    DRIVE / "models/athletiq_ball_detector_hard_finetuned_candidate.pt"
)
FINAL_CHAMPION = (
    DRIVE / "models/athletiq_ball_detector_champion_1280.pt"
)
EXPERIMENTS = DRIVE / "experiments/ball_detector_finalization"

assert ORIGINAL.is_file(), f"Missing original: {ORIGINAL}"
assert CANDIDATE.is_file(), f"Missing candidate: {CANDIDATE}"

# Promote the original checkpoint without changing its source file.
shutil.copy2(ORIGINAL, FINAL_CHAMPION)

decision = {
    "decision": "original_champion_selected_for_tracking",
    "date": datetime.now().isoformat(timespec="seconds"),
    "inference_image_size": 1280,
    "selected_checkpoint": str(FINAL_CHAMPION),
    "reason": (
        "Original champion achieved better hard-test recall, "
        "mAP50, and mAP50-95 than the fine-tuned candidate."
    ),
    "test_metrics": {
        "original_champion": {
            "precision": 0.5513,
            "recall": 0.4592,
            "map50": 0.4219,
            "map50_95": 0.1408,
            "inference_ms": 10.83
        },
        "finetuned_candidate": {
            "precision": 0.5897,
            "recall": 0.3670,
            "map50": 0.3645,
            "map50_95": 0.1230,
            "inference_ms": 10.80
        }
    },
    "candidate_retained": str(CANDIDATE),
    "original_source_retained": str(ORIGINAL),
    "test_split_note": (
        "Results are from the 1500-image hard test subset; "
        "do not treat them as full-dataset performance."
    )
}

decision_path = EXPERIMENTS / "production_champion_decision.json"
with open(decision_path, "w") as f:
    json.dump(decision, f, indent=4)

# Verify the selected model exists and has the same file size.
assert FINAL_CHAMPION.is_file()
assert FINAL_CHAMPION.stat().st_size == ORIGINAL.stat().st_size

print("PRODUCTION CHAMPION SELECTED")
print("Checkpoint:", FINAL_CHAMPION)
print("Resolution: 1280")
print("Decision report:", decision_path)
print("Original source remains unchanged.")
print("Fine-tuned candidate retained for research.")

PRODUCTION CHAMPION SELECTED
Checkpoint: /content/drive/MyDrive/AthletiQ/models/athletiq_ball_detector_champion_1280.pt
Resolution: 1280
Decision report: /content/drive/MyDrive/AthletiQ/experiments/ball_detector_finalization/production_champion_decision.json
Original source remains unchanged.
Fine-tuned candidate retained for research.
